In [1]:
import pandas as pd
import numpy as np
import pickle
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

In [2]:

def evaluate_portfolio(w, returns):
    daily_ret = returns @ w
    ann_return = daily_ret.mean() * 252
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    return ann_return, ann_vol, sharpe

In [3]:
def build_efficiency_table(
    results,
    returns_is,
    returns_oos,
    vol_levels
):
    rows = []

    # --- ESG portfolios ---
    for vol in vol_levels:
        # closest model-implied volatility
        vols = np.array([r["Volatility"] for r in results])
        idx = np.argmin(np.abs(vols - vol))
        w = results[idx]["Weights"]

        is_metrics  = evaluate_portfolio(w, returns_is)
        oos_metrics = evaluate_portfolio(w, returns_oos)

        rows.append({
            "Volatility Target": vol,
            "Return (IS)": is_metrics[0],
            "Return (OOS)": oos_metrics[0],
            "Volatility (OOS)": oos_metrics[1],
            "Sharpe (IS)": is_metrics[2],
            "Sharpe (OOS)": oos_metrics[2],
        })

 

    return pd.DataFrame(rows)


In [5]:
returns_oos = pd.read_parquet("data/returns_oos_2024.parquet")
returns_is_5y = pd.read_parquet("data/returns_5y.parquet")
returns_is_10y =  pd.read_parquet("data/returns_10y.parquet")

In [8]:
with open("results/optimized_weights_10y.pkl", "rb") as f:
    results_10y = pickle.load(f)

with open("results/optimized_weights_5y.pkl", "rb") as f:
    results_5y = pickle.load(f)

In [9]:
results_10y

[{'Volatility': 0.16000000488789265,
  'ESG': 85.89803005335574,
  'Weights': array([2.64276311e-09, 7.02361831e-09, 1.27933073e-08, 2.36526569e-08,
         9.09673129e-10, 1.65484047e-09, 2.59000728e-09, 4.47674553e-01,
         7.49974948e-09, 2.26299456e-09, 1.94465727e-09, 2.84717807e-01,
         1.73575398e-08, 1.15345311e-01, 5.93769698e-10, 1.71182139e-09,
         4.56687691e-09, 4.17222610e-09, 1.52262238e-01])},
 {'Volatility': 0.1700000203993497,
  'ESG': 86.85440270961296,
  'Weights': array([ 6.96996593e-09,  1.90462139e-08,  3.86181246e-08,  5.49331872e-08,
          8.94428550e-10,  3.23298221e-09,  5.67022215e-09,  4.42590937e-01,
          1.99448763e-08,  3.66700520e-09,  2.83473255e-09,  3.72598197e-01,
          2.54877954e-08,  1.54173123e-01, -5.84557601e-10,  2.68314190e-09,
          1.19476053e-08,  6.72234131e-09,  3.06375390e-02])},
 {'Volatility': 0.1800000001689806,
  'ESG': 87.48763772320267,
  'Weights': array([ 3.62487411e-11,  1.03220421e-10,  2.19036

In [6]:
returns_oos

Instrument,AAPL.OQ,AMGN.OQ,AXP.N,BA.N,CAT.N,CRM.N,CSCO.OQ,CVX.N,DIS.N,GS.N,HD.N,HON.OQ,IBM.N,INTC.OQ,JNJ.N,JPM.N,KO.N,MCD.N,MMM.N,MRK.N,MSFT.OQ,NKE.N,PG.N,TRV.N,UNH.N,V.N,VZ.N,WBA.OQ^H25,WMT.OQ
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2024-01-03,-0.007516,0.011035,-0.010624,-0.031677,-0.029152,-0.016891,0.0,0.018887,0.010309,-0.016908,-0.019961,-0.02162,-0.008707,-0.015815,0.006232,-0.004368,0.002338,-0.008961,-0.020295,0.013421,-0.000728,-0.023839,-0.006069,-0.000627,0.004975,-0.003444,0.007176,-0.041369,0.000063
2024-01-04,-0.012781,0.008214,0.004391,0.004214,0.006311,-0.002385,-0.00855,-0.011024,-0.011964,0.003034,0.000975,0.001807,0.004736,-0.003833,-0.002114,0.000467,-0.003341,-0.009042,0.003519,0.019329,-0.007203,-0.016866,0.005464,0.006461,0.006235,0.006298,0.005348,-0.052591,-0.009714
2024-01-05,-0.004021,-0.000561,0.010207,0.01644,0.009843,-0.000478,0.0002,-0.001727,0.003747,0.009072,0.012766,-0.006709,-0.010624,0.000427,0.003108,0.005005,-0.001507,-0.009471,0.003875,0.001793,-0.000517,-0.002153,-0.008309,0.002749,-0.014851,0.000308,0.020863,0.030447,-0.006678
2024-01-08,0.023887,0.025674,0.000793,-0.083731,0.011425,0.038091,0.00736,-0.006002,0.007125,0.006243,0.014446,-0.004284,0.012364,0.032728,0.002479,-0.001452,0.007347,0.010019,0.002483,0.001364,0.018696,0.014974,0.008578,-0.003944,-0.001602,0.010915,-0.002491,0.024488,0.009779
2024-01-09,-0.002266,-0.011713,-0.01298,-0.01425,0.000137,0.0018,-0.01096,-0.025747,-0.020749,-0.013254,-0.005014,-0.005095,-0.0066,-0.00829,0.000619,-0.007937,-0.001832,-0.003535,0.002202,0.008906,0.002931,-0.008529,0.004094,-0.000572,0.003442,0.003005,-0.02679,-0.01059,0.006676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.011413,0.001854,0.015475,0.009243,0.005949,0.004452,0.014643,0.006067,0.010359,0.020823,0.009359,0.007719,0.011113,0.009852,0.003985,0.01631,0.007347,0.009226,0.010643,0.000805,0.00933,0.000391,0.004925,0.006278,-0.000474,0.010755,-0.003511,-0.011898,0.025462
2024-12-26,0.003171,-0.004965,0.001745,0.005782,-0.001225,-0.007899,0.00217,0.000973,-0.000089,-0.00268,-0.002528,0.012849,0.002137,0.001959,-0.001853,0.00342,-0.004306,0.002751,0.006271,0.004214,-0.002781,0.001951,0.007196,0.003494,0.009929,0.000811,0.004012,0.051946,0.001186
2024-12-27,-0.013331,-0.002016,-0.009718,0.001883,-0.006175,-0.009615,-0.006188,0.000139,-0.008925,-0.008726,-0.005787,-0.010187,-0.009427,-0.006873,-0.003647,-0.008135,-0.00192,-0.004011,-0.007652,-0.001704,-0.017453,-0.006781,-0.003709,-0.009564,-0.002272,-0.007036,-0.001002,-0.006218,-0.012253


In [ ]:



def get_closest_result(results, vol_target):
    vols = np.array([r["Volatility"] for r in results])
    idx = np.argmin(np.abs(vols - vol_target))
    return results[idx]

idxes = [0, 4, 9, 14]

rows = []

for idx in idxes:
    
    # In-sample (IS)
    is_5y = evaluate_portfolio(results_5y[idx]["Weights"], returns_is_5y)
    is_10y = evaluate_portfolio(results_10y[idx]["Weights"], returns_is_10y)

    # out-of-sample (OOS) (2024)
    oos_5y = evaluate_portfolio(results_5y[idx]["Weights"], returns_oos)
    oos_10y = evaluate_portfolio(results_10y[idx]["Weights"], returns_oos)

    vol_levels = [0.16, 0.20, 0.25, 0.30]

table_5y = build_efficiency_table(
    results=results_5y,
    returns_is=returns_is_5y,
    returns_oos=returns_oos,
    vol_levels=vol_levels
)

table_10y = build_efficiency_table(
    results=results_10y,
    returns_is=returns_is_10y,
    returns_oos=returns_oos,
    vol_levels=vol_levels
)


ValueError: Dot product shape mismatch, (1259, 29) vs (19,)

In [ ]:
def build_benchmark_efficiency_table(w_is, w_oos, returns_is, returns_oos):
    is_metrics  = evaluate_portfolio(w_is, returns_is)
    oos_metrics = evaluate_portfolio(w_oos, returns_oos)

    return pd.DataFrame([{
        
        "Return (IS)": is_metrics[0],
        "Return (OOS)": oos_metrics[0],
        "Volatility (IS)": is_metrics[1],
        "Volatility (OOS)": oos_metrics[1],
        "Sharpe (IS)": is_metrics[2],
        "Sharpe (OOS)": oos_metrics[2],
    }])


In [ ]:
w_bench = pd.read_csv("data/benchmark_weights.csv").set_index("Instrument")['BenchWeight']

In [ ]:
bench_table_5y = build_benchmark_efficiency_table(
    w_is=w_bench,
    w_oos=w_bench,
    returns_is=returns_is_5y,
    returns_oos=returns_oos
)

bench_table_10y = build_benchmark_efficiency_table(
    w_is=w_bench,
    w_oos=w_bench,
    returns_is=returns_is_10y,
    returns_oos=returns_oos
)


In [ ]:
table_5y

,Volatility Target,Return (IS),Return (OOS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.16,0.036180,0.127777,0.105039,0.225335,1.216465
1,0.20,-0.012250,-0.139839,0.172062,-0.060896,-0.812725
2,0.25,-0.101288,-0.332617,0.279974,-0.402419,-1.188029
3,0.30,-0.172680,-0.728812,0.425553,-0.572701,-1.712624


In [ ]:
table_10y

,Volatility Target,Return (IS),Return (OOS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.16,0.041737,0.010543,0.137291,0.260236,0.076796
1,0.20,-0.000228,-0.207072,0.240900,-0.001133,-0.859574
2,0.25,-0.062990,-0.641336,0.398754,-0.251211,-1.608351
3,0.30,-0.078561,-1.043232,0.532634,-0.261313,-1.958627


In [ ]:
bench_table_10y

,Return (IS),Return (OOS),Volatility (IS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.15626,0.14511,0.194557,0.116252,0.803157,1.248233


In [ ]:
bench_table_5y


,Return (IS),Return (OOS),Volatility (IS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.185398,0.14511,0.228162,0.116252,0.812573,1.248233


In [ ]:
table_5y_fmt = table_5y.round(2)
table_10y_fmt = table_10y.round(2)
bench_table_5y_fmt = bench_table_5y.round(2)
bench_table_10y_fmt = bench_table_10y.round(2)

latex_5y = table_5y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of ESG-optimized portfolios (5-year estimation window).",
    label="tab:efficiency_5y",
    escape=False
)
latex_5y = latex_5y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)
latex_10y = table_10y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of ESG-optimized portfolios (10-year estimation window).",
    label="tab:efficiency_10y",
    escape=False
)
latex_10y = latex_10y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)
latex_bench_5y = bench_table_5y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of the benchmark portfolio (5-year estimation window).",
    label="tab:efficiency_benchmark_5y",
    escape=False
)
latex_bench_5y = latex_bench_5y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)

latex_bench_10y = bench_table_10y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of the benchmark portfolio (10-year estimation window).",
    label="tab:efficiency_benchmark_10y",
    escape=False
)
latex_bench_10y = latex_bench_10y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)

with open("results/efficiency_5y.tex", "w") as f:
    f.write(latex_5y)

with open("results/efficiency_10y.tex", "w") as f:
    f.write(latex_10y)

with open("results/efficiency_benchmark_5y.tex", "w") as f:
    f.write(latex_bench_5y)

with open("results/efficiency_benchmark_10y.tex", "w") as f:
    f.write(latex_bench_10y)
